# Pipeline ETL — Projeto de Integração de Dados (Zika Vírus SINAN: 2024, 2025 e 2026)
**Universidade Federal de Pernambuco (UFPE) — Centro de Informática (CIn)**  
**Disciplina:** Banco de Dados 

---
## Parte 1: Extração, Diagnóstico de Qualidade e Transformação em Memória (`E` + `T1`)
Esta seção realiza a leitura das 3 bases anuais brutas (`ZIKABR24.csv`, `ZIKABR25.csv`, `ZIKABR26.csv`), evidencia os problemas estruturais entre os anos (prints intermediários de *Antes vs. Depois*) e entrega o DataFrame consolidado e higienizado (`df_zika_limpo`) pronto para a montagem do Esquema Estrela.


In [2]:
import pandas as pd
import numpy as np
import os

# Configuração de exibição do Pandas para prints de evidência
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)


### 1.1. Extração e Consolidação Inicial (`Extract`)
Carregamento dos arquivos CSV (localmente da pasta `../dados/` ou direto do repositório) e adição da coluna de rastreabilidade `ARQUIVO_ORIGEM`.


In [4]:
# Ajuste automático do caminho (funciona tanto na raiz quanto dentro de /notebooks no Colab)
caminho_base = '../dados/' if os.path.exists('../dados/ZIKABR24.csv') else ('dados/' if os.path.exists('dados/ZIKABR24.csv') else './')

zika_24 = pd.read_csv(os.path.join(caminho_base, 'ZIKABR24.csv'), low_memory=False)
zika_25 = pd.read_csv(os.path.join(caminho_base, 'ZIKABR25.csv'), low_memory=False)
zika_26 = pd.read_csv(os.path.join(caminho_base, 'ZIKABR26.csv'), low_memory=False)

zika_24['ARQUIVO_ORIGEM'] = 'ZIKABR24.csv'
zika_25['ARQUIVO_ORIGEM'] = 'ZIKABR25.csv'
zika_26['ARQUIVO_ORIGEM'] = 'ZIKABR26.csv'

df_bruto = pd.concat([zika_24, zika_25, zika_26], ignore_index=True)

print(f"Registros ZIKABR24: {len(zika_24):,}")
print(f"Registros ZIKABR25: {len(zika_25):,}")
print(f"Registros ZIKABR26: {len(zika_26):,}")
print(f"Total Consolidado Bruto: {len(df_bruto):,} linhas x {df_bruto.shape[1]} colunas")


Registros ZIKABR24: 43,312
Registros ZIKABR25: 25,764
Registros ZIKABR26: 13,384
Total Consolidado Bruto: 82,460 linhas x 39 colunas


### 1.2. Diagnóstico das Inconsistências nas Bases Brutas (Evidência Intermediária — *ANTES*)
Antes de aplicar as transformações, inspecionamos as divergências reais entre as safras 2024, 2025 e 2026 documentadas no dicionário do projeto.


In [5]:
print("=== [ANTES] 1. Divergência no código ID_AGRAVO ('A92.' truncado no ES em 2024 vs 'A928') ===")
print(pd.crosstab(df_bruto['ARQUIVO_ORIGEM'], df_bruto['ID_AGRAVO'], dropna=False))

print("\n=== [ANTES] 2. Tipagem mista e zeros falsos em COUFINF (UF de Infecção) ===")
print("Tipos Python encontrados em COUFINF:", df_bruto['COUFINF'].apply(lambda x: type(x).__name__).value_counts().to_dict())
print("Ocorrências da string 'ES' em COUFINF:", (df_bruto['COUFINF'] == 'ES').sum())
print("Ocorrências de 0 falso em COUFINF:", ((df_bruto['COUFINF'] == 0) | (df_bruto['COUFINF'] == '0')).sum())

print("\n=== [ANTES] 3. Vazamento de Ano (NU_ANO vs Arquivo de Origem) ===")
print(pd.crosstab(df_bruto['ARQUIVO_ORIGEM'], df_bruto['NU_ANO'], dropna=False))

print("\n=== [ANTES] 4. Amostra de NU_IDADE_N codificado (4 dígitos do SINAN) ===")
print(df_bruto[['ARQUIVO_ORIGEM', 'ANO_NASC', 'NU_IDADE_N']].head(5))


=== [ANTES] 1. Divergência no código ID_AGRAVO ('A92.' truncado no ES em 2024 vs 'A928') ===
ID_AGRAVO       A92.   A928
ARQUIVO_ORIGEM             
ZIKABR24.csv    5427  37885
ZIKABR25.csv       0  25764
ZIKABR26.csv       0  13384

=== [ANTES] 2. Tipagem mista e zeros falsos em COUFINF (UF de Infecção) ===
Tipos Python encontrados em COUFINF: {'float': 79331, 'str': 3129}
Ocorrências da string 'ES' em COUFINF: 12
Ocorrências de 0 falso em COUFINF: 1434

=== [ANTES] 3. Vazamento de Ano (NU_ANO vs Arquivo de Origem) ===
NU_ANO           2024   2025   2026
ARQUIVO_ORIGEM                     
ZIKABR24.csv    43243     69      0
ZIKABR25.csv        6  25752      6
ZIKABR26.csv        0      0  13384

=== [ANTES] 4. Amostra de NU_IDADE_N codificado (4 dígitos do SINAN) ===
  ARQUIVO_ORIGEM  ANO_NASC  NU_IDADE_N
0   ZIKABR24.csv    1990.0        4033
1   ZIKABR24.csv    2024.0        2001
2   ZIKABR24.csv    1993.0        4030
3   ZIKABR24.csv    1998.0        4025
4   ZIKABR24.csv    1995.

### 1.3. Transformação e Higienização (`Transform — T1`)
Aplicação sequencial das 6 regras de limpeza e padronização de negócio:
1. **Padronização de `ID_AGRAVO`:** Unificação de `'A92.'` (5.427 registros de 2024) para `'A928'`.
2. **Higienização de `COUFINF`:** Conversão da sigla textual `'ES'` para o código IBGE `32` e substituição de `0` falso por `NaN`.
3. **Eliminação de Zeros Falsos (`0 -> NaN`):** Conversão do preenchimento `0` (inexistente no dicionário SINAN) para `NaN` em `CLASSI_FIN`, `CRITERIO`, `EVOLUCAO`, `ID_RG_RESI`, `NDUPLIC_N`, `IN_VINCULA`, `ID_OCUPA_N`, `COMUNINF`, `DOENCA_TRA`, `TP_SISTEMA`, `TPUNINOT` e `COPAISINF` (preservando `0 = Analfabeto` em `CS_ESCOL_N`).
4. **Decodificação da Idade (`NU_IDADE_N`):** Conversão do código composto de 4 dígitos (`1xxx`=Horas, `2xxx`=Dias, `3xxx`=Meses, `4xxx`=Anos) para `idade_anos` e categorização em `faixa_etaria`.
5. **Saneamento de Datas e Cálculo de Métricas Temporais:** Conversão para `datetime`, cálculo de `qt_dias_atraso_notif` (`DT_NOTIFIC - DT_SIN_PRI`) e `qt_dias_encerramento` (`DT_ENCERRA - DT_NOTIFIC`), expurgando os 73 registros com intervalos negativos (`< 0` dias, como `-674`) ou superiores a 1 ano (`> 365` dias).
6. **Enriquecimento Semântico e Flags Analíticas:** Mapeamento de códigos IBGE de UF para siglas e regiões do Brasil, tradução dos domínios numéricos (`ds_sexo`, `ds_raca_cor`, `ds_gestante`, `ds_escolaridade`, `ds_classificacao_final`, `ds_criterio_confirmacao`, `ds_evolucao_caso`) e criação das flags binárias (`fl_confirmado`, `fl_descartado`, `fl_gestante`, `fl_obito`, `fl_autoctone`).


In [6]:
df_zika_limpo = df_bruto.copy()

# 1. Padronização de ID_AGRAVO
df_zika_limpo['ID_AGRAVO'] = df_zika_limpo['ID_AGRAVO'].astype(str).str.strip().replace({'A92.': 'A928'})

# 2. Dicionários Geográficos (IBGE <-> Sigla UF e Região)
uf_sigla_to_ibge = {
    'RO': 11, 'AC': 12, 'AM': 13, 'RR': 14, 'PA': 15, 'AP': 16, 'TO': 17,
    'MA': 21, 'PI': 22, 'CE': 23, 'RN': 24, 'PB': 25, 'PE': 26, 'AL': 27,
    'SE': 28, 'BA': 29, 'MG': 31, 'ES': 32, 'RJ': 33, 'SP': 35, 'PR': 41,
    'SC': 42, 'RS': 43, 'MS': 50, 'MT': 51, 'GO': 52, 'DF': 53
}
ibge_to_uf_sigla = {v: k for k, v in uf_sigla_to_ibge.items()}
ibge_to_regiao = {
    11:'Norte', 12:'Norte', 13:'Norte', 14:'Norte', 15:'Norte', 16:'Norte', 17:'Norte',
    21:'Nordeste', 22:'Nordeste', 23:'Nordeste', 24:'Nordeste', 25:'Nordeste', 26:'Nordeste', 27:'Nordeste', 28:'Nordeste', 29:'Nordeste',
    31:'Sudeste', 32:'Sudeste', 33:'Sudeste', 35:'Sudeste',
    41:'Sul', 42:'Sul', 43:'Sul',
    50:'Centro-Oeste', 51:'Centro-Oeste', 52:'Centro-Oeste', 53:'Centro-Oeste'
}

def limpar_coufinf(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip().upper()
    if s in uf_sigla_to_ibge:
        return float(uf_sigla_to_ibge[s])
    try:
        num = float(s)
        return np.nan if num == 0.0 else num
    except:
        return np.nan

df_zika_limpo['COUFINF'] = df_zika_limpo['COUFINF'].apply(limpar_coufinf).astype('Int64')

# 3. Tratamento de Zeros Falsos (0 -> NaN)
colunas_zeros_falsos = [
    'CLASSI_FIN', 'CRITERIO', 'EVOLUCAO', 'ID_RG_RESI', 'NDUPLIC_N',
    'IN_VINCULA', 'ID_OCUPA_N', 'COMUNINF', 'DOENCA_TRA', 'TP_SISTEMA', 'TPUNINOT', 'COPAISINF'
]
for col in colunas_zeros_falsos:
    mask_zero = (df_zika_limpo[col] == 0) | (df_zika_limpo[col] == '0')
    df_zika_limpo.loc[mask_zero, col] = np.nan

# 4. Decodificação de NU_IDADE_N para idade em anos e faixa etária
def decodificar_idade_sinan(val):
    if pd.isna(val):
        return np.nan
    try:
        v = int(val)
        unidade, qtd = v // 1000, v % 1000
        if unidade == 4:
            return float(qtd) if qtd <= 120 else np.nan
        elif unidade == 3:
            return round(qtd / 12.0, 2)
        elif unidade == 2:
            return round(qtd / 365.0, 2)
        elif unidade == 1:
            return 0.0
        return np.nan
    except:
        return np.nan

def classificar_faixa_etaria(idade):
    if pd.isna(idade): return 'Ignorado'
    if idade < 1:      return '< 1 ano (Lactente)'
    if idade <= 9:     return '01 a 09 anos (Criança)'
    if idade <= 19:    return '10 a 19 anos (Adolescente)'
    if idade <= 39:    return '20 a 39 anos (Adulto Jovem)'
    if idade <= 59:    return '40 a 59 anos (Adulto)'
    return '60+ anos (Idoso)'

df_zika_limpo['idade_anos'] = df_zika_limpo['NU_IDADE_N'].apply(decodificar_idade_sinan)
df_zika_limpo['idade_anos_int'] = np.floor(df_zika_limpo['idade_anos']).astype('Int64')
df_zika_limpo['faixa_etaria'] = df_zika_limpo['idade_anos'].apply(classificar_faixa_etaria)

# 5. Conversão de Datas e Cálculo de Métricas Temporais Saneadas
colunas_data = ['DT_NOTIFIC', 'DT_SIN_PRI', 'DT_INVEST', 'DT_ENCERRA', 'DT_DIGITA', 'DT_OBITO']
for dcol in colunas_data:
    df_zika_limpo[dcol] = pd.to_datetime(df_zika_limpo[dcol], format='%Y-%m-%d', errors='coerce')

df_zika_limpo['ano_notificacao'] = df_zika_limpo['DT_NOTIFIC'].dt.year.astype('Int64')
df_zika_limpo['mes_notificacao'] = df_zika_limpo['DT_NOTIFIC'].dt.month.astype('Int64')
df_zika_limpo['trimestre_notificacao'] = df_zika_limpo['DT_NOTIFIC'].dt.quarter.astype('Int64')
df_zika_limpo['semana_epidemiologica'] = (df_zika_limpo['SEM_NOT'] % 100).astype('Int64')

atraso_dias = (df_zika_limpo['DT_NOTIFIC'] - df_zika_limpo['DT_SIN_PRI']).dt.days
encerra_dias = (df_zika_limpo['DT_ENCERRA'] - df_zika_limpo['DT_NOTIFIC']).dt.days

df_zika_limpo['qt_dias_atraso_notif'] = atraso_dias.where((atraso_dias >= 0) & (atraso_dias <= 365), np.nan)
df_zika_limpo['qt_dias_encerramento'] = encerra_dias.where((encerra_dias >= 0) & (encerra_dias <= 365), np.nan)

# 6. Decodificação de Domínios e Criação de Flags Analíticas
mapa_sexo = {'F': 'Feminino', 'M': 'Masculino', 'I': 'Ignorado'}
mapa_raca = {1: 'Branca', 2: 'Preta', 3: 'Amarela', 4: 'Parda', 5: 'Indígena', 9: 'Ignorado'}
mapa_gestante = {
    1: '1º Trimestre', 2: '2º Trimestre', 3: '3º Trimestre',
    4: 'Idade gestacional ignorada', 5: 'Não', 6: 'Não se aplica', 9: 'Ignorado'
}
mapa_escol = {
    0: 'Analfabeto', 1: '1ª a 4ª série incompleta', 2: '4ª série completa',
    3: '5ª à 8ª série incompleta', 4: 'Ensino fundamental completo',
    5: 'Ensino médio incompleto', 6: 'Ensino médio completo',
    7: 'Educação superior incompleta', 8: 'Educação superior completa',
    9: 'Ignorado', 10: 'Não se aplica'
}
mapa_classi = {1: 'Confirmado', 2: 'Descartado', 8: 'Inconclusivo'}
mapa_criterio = {1: 'Laboratorial', 2: 'Clínico-Epidemiológico'}
mapa_evolucao = {1: 'Cura', 2: 'Óbito pelo agravo', 3: 'Óbito por outras causas', 9: 'Ignorado'}

df_zika_limpo['ds_sexo'] = df_zika_limpo['CS_SEXO'].map(mapa_sexo).fillna('Ignorado')
df_zika_limpo['ds_raca_cor'] = df_zika_limpo['CS_RACA'].map(mapa_raca).fillna('Ignorado')
df_zika_limpo['ds_gestante'] = df_zika_limpo['CS_GESTANT'].map(mapa_gestante).fillna('Ignorado')
df_zika_limpo['ds_escolaridade'] = df_zika_limpo['CS_ESCOL_N'].map(mapa_escol).fillna('Ignorado')
df_zika_limpo['ds_classificacao_final'] = df_zika_limpo['CLASSI_FIN'].map(mapa_classi).fillna('Em Investigação / Ignorado')
df_zika_limpo['ds_criterio_confirmacao'] = df_zika_limpo['CRITERIO'].map(mapa_criterio).fillna('Não Informado')
df_zika_limpo['ds_evolucao_caso'] = df_zika_limpo['EVOLUCAO'].map(mapa_evolucao).fillna('Ignorado')

# Padronização de UF e Município (preenchendo 32 casos sem UF de residência pela UF de notificação)
df_zika_limpo['SG_UF_NOT'] = df_zika_limpo['SG_UF_NOT'].astype('Int64')
df_zika_limpo['SG_UF'] = df_zika_limpo['SG_UF'].fillna(df_zika_limpo['SG_UF_NOT']).astype('Int64')
df_zika_limpo['ID_MN_RESI'] = df_zika_limpo['ID_MN_RESI'].fillna(df_zika_limpo['ID_MUNICIP']).astype('Int64')

df_zika_limpo['sigla_uf_notificacao'] = df_zika_limpo['SG_UF_NOT'].map(ibge_to_uf_sigla)
df_zika_limpo['regiao_notificacao'] = df_zika_limpo['SG_UF_NOT'].map(ibge_to_regiao)
df_zika_limpo['sigla_uf_residencia'] = df_zika_limpo['SG_UF'].map(ibge_to_uf_sigla)
df_zika_limpo['regiao_residencia'] = df_zika_limpo['SG_UF'].map(ibge_to_regiao)

# Flags Binárias para a Tabela Fato
df_zika_limpo['fl_confirmado'] = (df_zika_limpo['CLASSI_FIN'] == 1).astype(int)
df_zika_limpo['fl_descartado'] = (df_zika_limpo['CLASSI_FIN'] == 2).astype(int)
df_zika_limpo['fl_gestante'] = df_zika_limpo['CS_GESTANT'].isin([1, 2, 3, 4]).astype(int)
df_zika_limpo['fl_obito'] = df_zika_limpo['EVOLUCAO'].isin([2, 3]).astype(int)
df_zika_limpo['fl_autoctone'] = (df_zika_limpo['TPAUTOCTO'] == 1).astype(int)

print("Transformação Parte 1 concluída! Shape de df_zika_limpo:", df_zika_limpo.shape)


Transformação Parte 1 concluída! Shape de df_zika_limpo: (82460, 64)


### 1.4. Validação Pós-Limpeza (Evidência Intermediária — *DEPOIS*)
Comprovação estatística das variáveis transformadas para inclusão no Relatório (Seção 4) e nos Slides.


In [7]:
print("=== [DEPOIS] 1. ID_AGRAVO 100% Padronizado ===")
print(df_zika_limpo['ID_AGRAVO'].value_counts(dropna=False))

print("\n=== [DEPOIS] 2. Estatísticas Descritivas das Métricas Contínuas Saneadas ===")
metricas = ['idade_anos', 'qt_dias_atraso_notif', 'qt_dias_encerramento']
resumo_stats = df_zika_limpo[metricas].agg(['count', 'mean', 'median', 'min', 'max', 'std']).T
print(resumo_stats.round(2))

print("\n=== [DEPOIS] 3. Distribuição por Faixa Etária Decodificada ===")
print(df_zika_limpo['faixa_etaria'].value_counts())

print("\n=== [DEPOIS] 4. Amostra das Colunas Prontas para o Esquema Estrela (Parte 2) ===")
colunas_contrato = [
    'DT_NOTIFIC', 'ano_notificacao', 'mes_notificacao', 'sigla_uf_residencia', 'ID_MN_RESI',
    'ds_sexo', 'idade_anos_int', 'faixa_etaria', 'ds_raca_cor', 'ds_gestante',
    'ds_classificacao_final', 'ds_criterio_confirmacao', 'ds_evolucao_caso',
    'qt_dias_atraso_notif', 'qt_dias_encerramento', 'fl_confirmado', 'fl_gestante', 'fl_obito'
]
print(df_zika_limpo[colunas_contrato].head(5))


=== [DEPOIS] 1. ID_AGRAVO 100% Padronizado ===
ID_AGRAVO
A928    82460
Name: count, dtype: int64

=== [DEPOIS] 2. Estatísticas Descritivas das Métricas Contínuas Saneadas ===
                        count   mean  median  min    max    std
idade_anos            82460.0  32.45    30.0  0.0  119.0  19.61
qt_dias_atraso_notif  82460.0   5.67     3.0  0.0  356.0  11.47
qt_dias_encerramento  80534.0  29.06    19.0  0.0  364.0  30.41

=== [DEPOIS] 3. Distribuição por Faixa Etária Decodificada ===
faixa_etaria
20 a 39 anos (Adulto Jovem)    31368
40 a 59 anos (Adulto)          19334
10 a 19 anos (Adolescente)     12597
60+ anos (Idoso)                8699
01 a 09 anos (Criança)          8254
< 1 ano (Lactente)              2208
Name: count, dtype: int64

=== [DEPOIS] 4. Amostra das Colunas Prontas para o Esquema Estrela (Parte 2) ===
  DT_NOTIFIC  ano_notificacao  mes_notificacao sigla_uf_residencia  ID_MN_RESI    ds_sexo  idade_anos_int                 faixa_etaria ds_raca_cor    ds_gestante 

---
## 🏛️ Parte 2: Modelagem Dimensional (Star Schema) e Carga no Banco (`T2` + `L`)
A partir do DataFrame `df_zika_limpo` (Parte 1), esta seção constrói as 5 dimensões com *surrogate keys*, monta a tabela fato `fato_notificacao_zika` e carrega o Esquema Estrela em um banco SQLite, comprovando o resultado com consultas SQL.


### 2.1. Preparação: chaves naturais e mapeamentos auxiliares (`T2`)
Antes de construir as dimensões, criamos a base de trabalho `df_base` a partir de `df_zika_limpo` (Parte 1), adicionando os nomes dos estados, os rótulos das colunas ainda codificadas (`TPAUTOCTO`, `DOENCA_TRA`) e substituindo nulos nas chaves naturais por um valor-sentinela. Assim, registros sem informação são ligados ao membro padrão `SK = 0` (*Não Informado*) em vez de se perderem no *merge*.

In [8]:
import sqlite3

# Nomes completos dos estados (código IBGE -> nome). Siglas e regiões já vêm da Parte 1.
ibge_to_nome_estado = {
    11:'Rondônia', 12:'Acre', 13:'Amazonas', 14:'Roraima', 15:'Pará', 16:'Amapá', 17:'Tocantins',
    21:'Maranhão', 22:'Piauí', 23:'Ceará', 24:'Rio Grande do Norte', 25:'Paraíba', 26:'Pernambuco',
    27:'Alagoas', 28:'Sergipe', 29:'Bahia', 31:'Minas Gerais', 32:'Espírito Santo', 33:'Rio de Janeiro',
    35:'São Paulo', 41:'Paraná', 42:'Santa Catarina', 43:'Rio Grande do Sul',
    50:'Mato Grosso do Sul', 51:'Mato Grosso', 52:'Goiás', 53:'Distrito Federal'
}
mapa_autoctonia = {1: 'Autóctone', 2: 'Importado', 3: 'Indeterminado / Ignorado'}
mapa_doenca_trab = {1: 'Sim', 2: 'Não', 9: 'Ignorado'}

df_base = df_zika_limpo.copy()

# Colunas descritivas que ainda faltam para as dimensões
df_base['nome_estado_residencia'] = df_base['SG_UF'].map(ibge_to_nome_estado).fillna('Não Informado')
df_base['ds_autoctonia']    = df_base['TPAUTOCTO'].map(mapa_autoctonia).fillna('Indeterminado / Ignorado')
df_base['ds_doenca_trab']   = df_base['DOENCA_TRA'].map(mapa_doenca_trab).fillna('Ignorado')
df_base['regiao_residencia'] = df_base['regiao_residencia'].fillna('Não Informado')
df_base['sigla_uf_residencia'] = df_base['sigla_uf_residencia'].fillna('NI')
df_base['sigla_uf_notificacao'] = df_base['sigla_uf_notificacao'].fillna('NI')

# Chaves naturais numéricas: nulo -> -1 (sentinela), para o merge não descartar linhas
for col in ['ID_MN_RESI', 'ID_RG_RESI', 'ID_UNIDADE', 'SG_UF', 'SG_UF_NOT', 'ID_MUNICIP', 'ID_REGIONA', 'idade_anos_int']:
    df_base[col + '_k'] = df_base[col].astype('float').fillna(-1).astype(int)

print(f"df_base pronto: {df_base.shape[0]:,} linhas")

df_base pronto: 82,460 linhas


### 2.2. Construção das 5 Dimensões com Surrogate Keys (`drop_duplicates` + chaves substitutas)
Para cada dimensão: (i) extraímos as combinações únicas dos atributos com `.drop_duplicates()`, (ii) geramos a *surrogate key* sequencial a partir de 1 (a `SK = 0` fica reservada ao registro *Não Informado*, inserido pelo DDL em `sql/01_script_ddl_do_esquema_estrela.sql`). Em `dim_tempo` a chave segue o formato `YYYYMMDD`, conforme o DDL.

In [9]:
def semana_epidemiologica(dt):
    """Semana epidemiológica (domingo a sábado; semana 1 contém 4 de janeiro, regra CDC/MMWR usada pelo SINAN)."""
    dt = pd.Timestamp(dt)
    def inicio_semana1(ano):
        jan4 = pd.Timestamp(year=ano, month=1, day=4)
        return jan4 - pd.Timedelta(days=(jan4.weekday() + 1) % 7)
    ano = dt.year
    if dt >= inicio_semana1(ano + 1):
        ano += 1
    elif dt < inicio_semana1(ano):
        ano -= 1
    return (dt - inicio_semana1(ano)).days // 7 + 1

nomes_mes = {1:'Janeiro', 2:'Fevereiro', 3:'Março', 4:'Abril', 5:'Maio', 6:'Junho',
             7:'Julho', 8:'Agosto', 9:'Setembro', 10:'Outubro', 11:'Novembro', 12:'Dezembro'}

# ---------- DIM_TEMPO (datas de notificação + datas de primeiros sintomas) ----------
datas = pd.concat([df_base['DT_NOTIFIC'], df_base['DT_SIN_PRI']]).dropna().drop_duplicates().sort_values()
dim_tempo = pd.DataFrame({'data_completa': datas.values})
dim_tempo['sk_tempo'] = dim_tempo['data_completa'].dt.strftime('%Y%m%d').astype(int)
dim_tempo['ano'] = dim_tempo['data_completa'].dt.year
dim_tempo['mes'] = dim_tempo['data_completa'].dt.month
dim_tempo['nome_mes'] = dim_tempo['mes'].map(nomes_mes)
dim_tempo['trimestre'] = dim_tempo['data_completa'].dt.quarter
dim_tempo['semana_epidemiologica'] = dim_tempo['data_completa'].apply(semana_epidemiologica)
dim_tempo['periodo_sazonal'] = np.where(dim_tempo['mes'] <= 5, 'Alta Transmissão (Jan-Mai)', 'Baixa Transmissão (Jun-Dez)')
dim_tempo['data_completa'] = dim_tempo['data_completa'].dt.strftime('%Y-%m-%d')   # DATE como texto ISO no SQLite
dim_tempo = dim_tempo[['sk_tempo','data_completa','ano','mes','nome_mes','trimestre','semana_epidemiologica','periodo_sazonal']]

# ---------- DIM_LOCALIDADE (residência do paciente) ----------
cols_loc = ['SG_UF_k', 'sigla_uf_residencia', 'nome_estado_residencia', 'regiao_residencia', 'ID_MN_RESI_k', 'ID_RG_RESI_k']
dim_localidade = df_base[cols_loc].drop_duplicates().reset_index(drop=True)
dim_localidade.insert(0, 'sk_localidade', np.arange(1, len(dim_localidade) + 1))
dim_localidade.columns = ['sk_localidade', 'cod_uf_ibge', 'sigla_uf', 'nome_estado', 'regiao_brasil', 'cod_municipio_ibge6', 'cod_regional_saude']

# ---------- DIM_UNIDADE_NOTIFICADORA ----------
cols_uni = ['ID_UNIDADE_k', 'SG_UF_NOT_k', 'sigla_uf_notificacao', 'ID_MUNICIP_k', 'ID_REGIONA_k']
dim_unidade = df_base[cols_uni].drop_duplicates().reset_index(drop=True)
dim_unidade.insert(0, 'sk_unidade', np.arange(1, len(dim_unidade) + 1))
dim_unidade.columns = ['sk_unidade', 'cod_unidade_cnes', 'cod_uf_notificacao', 'sigla_uf_notificacao', 'cod_municipio_notificacao', 'cod_regional_notificacao']

# ---------- DIM_PACIENTE ----------
cols_pac = ['CS_SEXO', 'ds_sexo', 'idade_anos_int_k', 'faixa_etaria', 'ds_raca_cor', 'ds_gestante', 'ds_escolaridade']
dim_paciente = df_base[cols_pac].drop_duplicates().reset_index(drop=True)
dim_paciente.insert(0, 'sk_paciente', np.arange(1, len(dim_paciente) + 1))
dim_paciente.columns = ['sk_paciente', 'sexo', 'descricao_sexo', 'idade_anos', 'faixa_etaria', 'raca_cor', 'status_gestante', 'escolaridade']

# ---------- DIM_INVESTIGACAO_DESFECHO ----------
cols_des = ['ds_classificacao_final', 'ds_criterio_confirmacao', 'ds_evolucao_caso', 'ds_autoctonia', 'ds_doenca_trab']
dim_desfecho = df_base[cols_des].drop_duplicates().reset_index(drop=True)
dim_desfecho.insert(0, 'sk_desfecho', np.arange(1, len(dim_desfecho) + 1))
dim_desfecho.columns = ['sk_desfecho', 'classificacao_final', 'criterio_confirmacao', 'evolucao_caso', 'tipo_autoctonia', 'doenca_trabalho']

for nome, d in [('dim_tempo', dim_tempo), ('dim_localidade', dim_localidade), ('dim_unidade_notificadora', dim_unidade),
                ('dim_paciente', dim_paciente), ('dim_investigacao_desfecho', dim_desfecho)]:
    print(f"{nome:<28} {len(d):>7,} linhas x {d.shape[1]} colunas")

dim_tempo                      1,002 linhas x 8 colunas
dim_localidade                 2,574 linhas x 7 colunas
dim_unidade_notificadora       6,630 linhas x 6 colunas
dim_paciente                   9,023 linhas x 8 colunas
dim_investigacao_desfecho        147 linhas x 6 colunas


### 2.3. Evidência intermediária: amostra das dimensões

In [10]:
print("=== dim_tempo ==="); print(dim_tempo.head(5))
print("\n=== dim_localidade ==="); print(dim_localidade.head(5))
print("\n=== dim_unidade_notificadora ==="); print(dim_unidade.head(5))
print("\n=== dim_paciente ==="); print(dim_paciente.head(5))
print("\n=== dim_investigacao_desfecho ==="); print(dim_desfecho.head(5))
print("\nUFs mapeadas em dim_localidade:", dim_localidade.loc[dim_localidade['cod_uf_ibge'] > 0, 'sigla_uf'].nunique(), "(esperado: 27)")

=== dim_tempo ===
   sk_tempo data_completa   ano  mes  nome_mes  trimestre  semana_epidemiologica              periodo_sazonal
0  20231231    2023-12-31  2023   12  Dezembro          4                      1  Baixa Transmissão (Jun-Dez)
1  20240101    2024-01-01  2024    1   Janeiro          1                      1   Alta Transmissão (Jan-Mai)
2  20240102    2024-01-02  2024    1   Janeiro          1                      1   Alta Transmissão (Jan-Mai)
3  20240103    2024-01-03  2024    1   Janeiro          1                      1   Alta Transmissão (Jan-Mai)
4  20240104    2024-01-04  2024    1   Janeiro          1                      1   Alta Transmissão (Jan-Mai)

=== dim_localidade ===
   sk_localidade  cod_uf_ibge sigla_uf     nome_estado regiao_brasil  cod_municipio_ibge6  cod_regional_saude
0              1           32       ES  Espírito Santo       Sudeste               320010                  -1
1              2           32       ES  Espírito Santo       Sudeste          

### 2.4. Montagem da Tabela Fato (`fato_notificacao_zika`)
O `merge` do DataFrame principal com cada dimensão (usando a chave natural, com `how='left'`) troca os atributos descritivos pelas chaves estrangeiras. Datas ausentes (`DT_SIN_PRI`) recebem `FK = 0`. A cada passo conferimos que o número de linhas **não mudou** (um *merge* mal feito duplicaria registros). As métricas e *flags* vêm prontas da Parte 1; `fl_obito_zika` segue o dicionário do projeto (`EVOLUCAO = 2`, óbito pelo agravo), diferente de `fl_obito` da Parte 1, que também conta óbitos por outras causas.

In [11]:
fato = df_base.copy()
n_inicial = len(fato)

# Chaves de data (YYYYMMDD) - ausente -> 0
fato['fk_tempo_notificacao'] = fato['DT_NOTIFIC'].dt.strftime('%Y%m%d').astype(float).fillna(0).astype(int)
fato['fk_tempo_sintomas']    = fato['DT_SIN_PRI'].dt.strftime('%Y%m%d').astype(float).fillna(0).astype(int)

# Merge com as 4 dimensões que têm chave substituta sequencial
fato = fato.merge(dim_localidade.rename(columns={
        'cod_uf_ibge': 'SG_UF_k', 'sigla_uf': 'sigla_uf_residencia', 'nome_estado': 'nome_estado_residencia',
        'regiao_brasil': 'regiao_residencia', 'cod_municipio_ibge6': 'ID_MN_RESI_k', 'cod_regional_saude': 'ID_RG_RESI_k'}),
    on=cols_loc, how='left').rename(columns={'sk_localidade': 'fk_localidade_residencia'})

fato = fato.merge(dim_unidade.rename(columns={
        'cod_unidade_cnes': 'ID_UNIDADE_k', 'cod_uf_notificacao': 'SG_UF_NOT_k', 'cod_municipio_notificacao': 'ID_MUNICIP_k',
        'cod_regional_notificacao': 'ID_REGIONA_k'}),
    on=cols_uni, how='left').rename(columns={'sk_unidade': 'fk_unidade_notificadora'})

fato = fato.merge(dim_paciente.rename(columns={
        'sexo': 'CS_SEXO', 'descricao_sexo': 'ds_sexo', 'idade_anos': 'idade_anos_int_k', 'raca_cor': 'ds_raca_cor',
        'status_gestante': 'ds_gestante', 'escolaridade': 'ds_escolaridade'}),
    on=cols_pac, how='left').rename(columns={'sk_paciente': 'fk_paciente'})

fato = fato.merge(dim_desfecho.rename(columns={
        'classificacao_final': 'ds_classificacao_final', 'criterio_confirmacao': 'ds_criterio_confirmacao',
        'evolucao_caso': 'ds_evolucao_caso', 'tipo_autoctonia': 'ds_autoctonia', 'doenca_trabalho': 'ds_doenca_trab'}),
    on=cols_des, how='left').rename(columns={'sk_desfecho': 'fk_desfecho'})

assert len(fato) == n_inicial, "Merge duplicou ou perdeu linhas!"

# Métricas e flags
fato['qt_notificacoes'] = 1
fato['fl_obito_zika'] = (fato['EVOLUCAO'] == 2).astype(int)
fato.insert(0, 'sk_notificacao', np.arange(1, len(fato) + 1))

colunas_fato = ['sk_notificacao', 'fk_tempo_notificacao', 'fk_tempo_sintomas', 'fk_localidade_residencia',
                'fk_unidade_notificadora', 'fk_paciente', 'fk_desfecho', 'qt_notificacoes',
                'fl_confirmado', 'fl_descartado', 'fl_gestante', 'fl_obito_zika',
                'qt_dias_atraso_notif', 'qt_dias_encerramento']
fato_notificacao_zika = fato[colunas_fato].copy()
fato_notificacao_zika[['qt_dias_atraso_notif', 'qt_dias_encerramento']] = fato_notificacao_zika[['qt_dias_atraso_notif', 'qt_dias_encerramento']].astype('Int64')

# Validação: nenhuma FK pode ficar nula
fks = [c for c in colunas_fato if c.startswith('fk_')]
print("FKs nulas na fato:", fato_notificacao_zika[fks].isna().sum().to_dict())
print(f"Linhas na fato: {len(fato_notificacao_zika):,} (df_zika_limpo: {n_inicial:,})")
print(fato_notificacao_zika.head(5))

FKs nulas na fato: {'fk_tempo_notificacao': 0, 'fk_tempo_sintomas': 0, 'fk_localidade_residencia': 0, 'fk_unidade_notificadora': 0, 'fk_paciente': 0, 'fk_desfecho': 0}
Linhas na fato: 82,460 (df_zika_limpo: 82,460)
   sk_notificacao  fk_tempo_notificacao  fk_tempo_sintomas  fk_localidade_residencia  fk_unidade_notificadora  fk_paciente  fk_desfecho  qt_notificacoes  fl_confirmado  fl_descartado  fl_gestante  fl_obito_zika  qt_dias_atraso_notif  qt_dias_encerramento
0               1              20240108           20240107                         1                        1            1            1                1              0              1            1              0                     1                     8
1               2              20240108           20240108                         2                        2            2            1                1              0              1            0              0                     0                     1
2               3   

### 2.5. Carga no Banco (`Load`) — SQLite + `to_sql()`
O banco é criado executando o DDL do repositório (`sql/01_script_ddl_do_esquema_estrela.sql`), que define as PKs, FKs e os registros *Não Informado* (`SK = 0`). Em seguida, o `to_sql(if_exists='append')` carrega as 5 dimensões e a fato na ordem correta de dependência (dimensões primeiro). O SQLite já vem no Python, então não há nada para instalar no Colab.

In [12]:
caminho_ddl = next((p for p in ['../sql/01_script_ddl_do_esquema_estrela.sql', 'sql/01_script_ddl_do_esquema_estrela.sql',
                                '01_script_ddl_do_esquema_estrela.sql'] if os.path.exists(p)), None)
assert caminho_ddl, "DDL não encontrado: rode a célula de clonagem do repositório (Colab) ou abra o notebook dentro do repo."

DB_PATH = 'dw_zika_etl.db'
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
conn.execute('PRAGMA foreign_keys = ON;')          # garante a integridade referencial na carga
with open(caminho_ddl, encoding='utf-8') as f:
    conn.executescript(f.read())

# Sentinela -1 (usada só para o merge) vira NULL de verdade nas dimensões
def sentinela_para_null(df):
    df = df.copy()
    for c in df.select_dtypes(include='number').columns:
        if c.startswith('sk_'):
            continue
        if (df[c] == -1).any():
            df[c] = df[c].where(df[c] != -1).astype('Int64')
    return df

dim_localidade_db = sentinela_para_null(dim_localidade)
dim_unidade_db    = sentinela_para_null(dim_unidade)
dim_paciente_db   = sentinela_para_null(dim_paciente)

# Dimensões primeiro, fato por último
dim_tempo.to_sql('dim_tempo', conn, if_exists='append', index=False)
dim_localidade_db.to_sql('dim_localidade', conn, if_exists='append', index=False)
dim_unidade_db.to_sql('dim_unidade_notificadora', conn, if_exists='append', index=False)
dim_paciente_db.to_sql('dim_paciente', conn, if_exists='append', index=False)
dim_desfecho.to_sql('dim_investigacao_desfecho', conn, if_exists='append', index=False)
fato_notificacao_zika.to_sql('fato_notificacao_zika', conn, if_exists='append', index=False, chunksize=5000)
conn.commit()
print(f"Carga concluída em '{DB_PATH}'.")

Carga concluída em 'dw_zika_etl.db'.


### 2.6. Comprovação no SGBD (consultas de verificação)

In [13]:
def q(sql):
    return pd.read_sql_query(sql, conn)

print("=== 1. Contagem de linhas por tabela ===")
print(q("""
SELECT 'dim_tempo' AS tabela, COUNT(*) AS linhas FROM dim_tempo
UNION ALL SELECT 'dim_localidade', COUNT(*) FROM dim_localidade
UNION ALL SELECT 'dim_unidade_notificadora', COUNT(*) FROM dim_unidade_notificadora
UNION ALL SELECT 'dim_paciente', COUNT(*) FROM dim_paciente
UNION ALL SELECT 'dim_investigacao_desfecho', COUNT(*) FROM dim_investigacao_desfecho
UNION ALL SELECT 'fato_notificacao_zika', COUNT(*) FROM fato_notificacao_zika
"""))

print("\n=== 2. Integridade referencial (deve retornar vazio) ===")
print(q("PRAGMA foreign_key_check;"))

print("\n=== 3. JOIN de teste: notificações por ano, trimestre e classificação final ===")
print(q("""
SELECT t.ano, d.classificacao_final, COUNT(*) AS notificacoes
FROM fato_notificacao_zika f
JOIN dim_tempo t ON t.sk_tempo = f.fk_tempo_notificacao
JOIN dim_investigacao_desfecho d ON d.sk_desfecho = f.fk_desfecho
GROUP BY t.ano, d.classificacao_final
ORDER BY t.ano, notificacoes DESC
"""))

print("\n=== 4. JOIN de teste: Top 10 UFs de residência (casos, confirmados e gestantes) ===")
print(q("""
SELECT l.sigla_uf, l.regiao_brasil, COUNT(*) AS notificacoes,
       SUM(f.fl_confirmado) AS confirmados, SUM(f.fl_gestante) AS gestantes
FROM fato_notificacao_zika f
JOIN dim_localidade l ON l.sk_localidade = f.fk_localidade_residencia
GROUP BY l.sigla_uf, l.regiao_brasil
ORDER BY notificacoes DESC LIMIT 10
"""))

print("\n=== 5. JOIN de teste: perfil do paciente x classificação (faixa etária) ===")
print(q("""
SELECT p.faixa_etaria, COUNT(*) AS notificacoes, ROUND(AVG(f.qt_dias_atraso_notif), 2) AS media_atraso_dias
FROM fato_notificacao_zika f
JOIN dim_paciente p ON p.sk_paciente = f.fk_paciente
GROUP BY p.faixa_etaria ORDER BY notificacoes DESC
"""))

=== 1. Contagem de linhas por tabela ===
                      tabela  linhas
0                  dim_tempo    1003
1             dim_localidade    2575
2   dim_unidade_notificadora    6631
3               dim_paciente    9023
4  dim_investigacao_desfecho     147
5      fato_notificacao_zika   82460

=== 2. Integridade referencial (deve retornar vazio) ===
Empty DataFrame
Columns: [table, rowid, parent, fkid]
Index: []

=== 3. JOIN de teste: notificações por ano, trimestre e classificação final ===
     ano         classificacao_final  notificacoes
0   2024                  Descartado         38135
1   2024                Inconclusivo          2509
2   2024                  Confirmado          2068
3   2024  Em Investigação / Ignorado           537
4   2025                  Descartado         22156
5   2025                  Confirmado          1875
6   2025                Inconclusivo          1353
7   2025  Em Investigação / Ignorado           437
8   2026                  Descartado  

### 2.7. Reconciliação final: DataFrame x Banco
Garante que a carga não perdeu nem inventou dados: totais e somas das *flags* no banco precisam bater com o DataFrame de origem.

In [14]:
sql_tot = q("""SELECT COUNT(*) AS total, SUM(fl_confirmado) AS confirmados, SUM(fl_descartado) AS descartados,
                      SUM(fl_gestante) AS gestantes, SUM(fl_obito_zika) AS obitos_zika FROM fato_notificacao_zika""").iloc[0]
pandas_tot = {'total': len(df_zika_limpo), 'confirmados': int(df_zika_limpo['fl_confirmado'].sum()),
              'descartados': int(df_zika_limpo['fl_descartado'].sum()), 'gestantes': int(df_zika_limpo['fl_gestante'].sum()),
              'obitos_zika': int((df_zika_limpo['EVOLUCAO'] == 2).sum())}
comparativo = pd.DataFrame({'DataFrame (Pandas)': pandas_tot, 'Banco (SQLite)': sql_tot.astype(int).to_dict()})
comparativo['OK?'] = comparativo['DataFrame (Pandas)'] == comparativo['Banco (SQLite)']
print(comparativo)
assert comparativo['OK?'].all(), "Divergência entre DataFrame e banco!"
print("\n✅ Pipeline ETL concluído: Esquema Estrela carregado e reconciliado.")
conn.close()

             DataFrame (Pandas)  Banco (SQLite)   OK?
total                     82460           82460  True
confirmados                4619            4619  True
descartados               71540           71540  True
gestantes                  7185            7185  True
obitos_zika                   1               1  True

✅ Pipeline ETL concluído: Esquema Estrela carregado e reconciliado.
